# Block 5 — the CatBoost pair model

**Reads:** Block 3 pair features (`workspace/block3/v1/features/…`, ~75 hand-crafted features) and
Block 4 context features (`workspace/block4/ctx/v1/…`).
**Writes:** `workspace/block5/final/scores/<split>/<ROLE>/<COUNTRY>.parquet` — one match probability
`p` per candidate pair for CALIBRATION and HOLDOUT (INDIA, US) and TEST (INDIA, US, FRANCE) — plus
the model file and `features_final.json`.

Fixed design:

* **One CatBoost model**, trained on **TREE-FIT**, early-stopped on **TREE-DEV** (log-loss). No
  ensembling. CALIBRATION, HOLDOUT and TEST are never trained on.
* **Features** = Block 3 selected features + Block 4 context up to level **L2** (cumulative BASE, L0,
  L1, L2 — L2, the record-claimant level, won the development ablation: BASE 0.9855, L0 0.9855,
  L1 0.9856, L2 0.9871 S1-macro F0.5 on a 20% TREE-DEV subsample). Signals needing a second
  cross-encoder do not exist here and are dropped. A **train-vs-test parity check** (KS > 0.10 or
  NaN-rate gap > 5pp between HOLDOUT and TEST) removes any signal whose distribution shifts; its
  result is printed below.
* **Hyper-parameters**, first hit wins: `CB_PARAMS` → `block5/final/hpo/best.json` →
  `src/catboost_params.json` → the newest older `block5/*/hpo/best.json` → `b5.CB_DEFAULT`
  (depth 8, lr 0.08, l2 3.0, random strength 1.0, bagging temperature 0.5, 254 borders — what the
  submitted model used). `RUN_HPO = True` runs a time-boxed optuna search when nothing tuned exists.
  `iterations` is a 6000 ceiling; early stopping picks the real count.
* France is **never** trained on (no labels) — the same model scores it, with `country` as a
  categorical feature.

Needs Blocks 1–4 finished. CatBoost GPU is much faster; CPU works and is slower.

## Setup (identical first code cell in every notebook)

`BASE` is the folder that holds `src/` — found by walking up from the working directory, or taken
from `$BER_HOME`. Everything then follows: `DATA = BASE/data` (read-only, supplied by the grader),
`WORKSPACE = BASE/workspace` (everything this pipeline writes). `$BER_WORKSPACE` is set to `WORKSPACE`
**before any module is imported**, because `b2_common` / `b2_prod` compute their paths at import
time. `$HF_HOME` keeps model downloads inside the workspace. Nothing outside `BASE` is read or
written except temporary scratch files.

In [ ]:
# Setup — repository layout, workspace, module path
import os
import sys
import tempfile
from pathlib import Path


def find_base(start=None):
    """The repository folder that holds src/. Override with $BER_HOME."""
    if os.environ.get("BER_HOME"):
        return Path(os.environ["BER_HOME"]).expanduser().resolve()
    here = Path(start or os.getcwd()).resolve()
    for d in (here, *here.parents):
        if (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"no folder containing src/ at or above {here}; set BER_HOME")


BASE = find_base()
SRC, DATA, WORKSPACE = BASE / "src", BASE / "data", BASE / "workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
ROOT = str(WORKSPACE)

os.environ["BER_WORKSPACE"] = ROOT                       # b2_common / b2_prod read this at import time
os.environ["HF_HOME"] = str(WORKSPACE / "hf_cache")   # model downloads stay inside the workspace
WORK = tempfile.gettempdir()                          # scratch for large local-first writes
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("BASE     :", BASE)
print("DATA     :", DATA, "" if DATA.is_dir() else "(MISSING — supplied by the grader)")
print("WORKSPACE:", WORKSPACE)

import gc
import glob
import importlib
import json
import shutil
import subprocess
import time

import numpy as np

def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def jdump(obj, path):
    """JSON with numpy types converted, written to a tmp file in the same folder then renamed."""
    def _d(o):
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.bool_):
            return bool(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)
    path = str(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    json.dump(obj, open(tmp, "w"), indent=1, default=_d)
    os.replace(tmp, path)


def require(*mods):
    """Fail early with one clear message when an expected package is missing."""
    missing = []
    for m in mods:
        try:
            importlib.import_module(m)
        except ImportError:
            missing.append(m)
    if missing:
        raise ImportError(f"missing packages: {missing} — pip install -r ../requirements.txt")

In [ ]:
# Stage 0 — config
B3_VERSION = "v1"
CTX_VERSION = "v1"
B5_VERSION = "final"               # outputs go to workspace/block5/final
TRAIN_C = ["INDIA", "US"]
TEST_C = ["INDIA", "US", "FRANCE"]
SEED = 0
ITER_CEILING = 6000
CB_TASK_TYPE = None                # None = GPU if nvidia-smi works, else CPU
RUN_HPO = False                    # True = time-boxed CatBoost search when no tuned params exist
HPO_MINUTES = 60
HPO_MAX_TRIALS = 25
HPO_FIT_FRAC = 0.30                # share of TREE-FIT S1s used inside the search (TREE-DEV always in full)
FORCE_RETRAIN = False
FORCE_RESCORE = False
PARITY_EXPECTED_DROP = ["rec_gap_best", "rec_share"]     # what the submitted run's parity check dropped

# Optional override of the six CatBoost keys. None -> b5.CB_DEFAULT (what the submitted model used);
# otherwise a dict here, or catboost_params.json in src/.
CB_PARAMS = None

In [ ]:
# Setup 2 — imports
import pandas as pd
require("catboost")
import er_eval as ev
import b2_prod as b2p
import b3_features as b3
import b4_ctx as cx
import b5_catboost as b5

ev.set_root(ROOT)
K_FINAL = int(b2p.load_merged_manifest()["k_final"])

# Block 3's selected feature list (which of the ~75 features Block 3 kept) — required, never guessed
_cands = [WORKSPACE / "block3" / B3_VERSION / "reports" / "selected_features.json",
          WORKSPACE / "block3" / B3_VERSION / "selected_features.json",
          SRC / "selected_features.json"]
B3_SELECTED = next((str(p) for p in _cands if p.exists()), None)
if B3_SELECTED is None:
    raise FileNotFoundError(f"selected_features.json not found; looked in {[str(p) for p in _cands]} "
                            "- run Block 3, or copy its output next to the code")
if CB_TASK_TYPE is None:
    try:
        CB_TASK_TYPE = "GPU" if subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0 else "CPU"
    except FileNotFoundError:
        CB_TASK_TYPE = "CPU"
b5.configure(root=ROOT, b3_version=B3_VERSION, b3_selected=B3_SELECTED, ctx_version=CTX_VERSION,
             out_version=B5_VERSION, task_type=CB_TASK_TYPE)
CATS = b3.CAT_FEATURES
os.makedirs(b5.P(), exist_ok=True)
log(f"K_FINAL={K_FINAL} | Block 3 selected list: {B3_SELECTED} | CatBoost {CB_TASK_TYPE} | "
    f"{b5.B5_CODE_VERSION} / {cx.CTX_CODE_VERSION}")

In [ ]:
# Stage 1 — preflight: everything Block 5 reads must be committed
def _ctx_ok(split, role, cs):
    return all(os.path.exists(cx.ctx_path(ROOT, CTX_VERSION, split, role, c).replace(".parquet", ".done.json")) for c in cs)


need = [("train", r, TRAIN_C) for r in ("TREE-FIT", "TREE-DEV", "CALIBRATION", "HOLDOUT")] + [("test", "TEST", TEST_C)]
for sp, r, cs in need:
    n3 = {c: len(b3.committed_parts(B3_VERSION, sp, [r], [c])) for c in cs}
    print(f"{sp}/{r:<11} block3 parts {n3} | block4/ctx {'OK' if _ctx_ok(sp, r, cs) else 'MISSING'}")
    assert all(n3.values()) and _ctx_ok(sp, r, cs), f"{sp}/{r}: run Block 3 / Block 4 first"
HAS_V1 = cx.ctx_has_v1(ROOT, CTX_VERSION, "train", "TREE-FIT", TRAIN_C)
print("second cross-encoder (V1) signals present:", HAS_V1)

## Feature set
Block 3 features + Block 4 levels BASE → L2 (cumulative), minus V1-dependent signals, minus signals
that fail the train-vs-test parity check (HOLDOUT vs TEST, same countries). Both lists are printed so
a re-run can be compared with the submitted run.

In [ ]:
# Stage 2 — features
B3F = b5.b3_feature_list(b3)
LEVELS = b5.ctx_levels(HAS_V1)                                  # drops V1-dependent signals when there is no V1
FSET_L2 = b5.variant_features(B3F, LEVELS, "L2")                # Block 3 + BASE + L0 + L1 + L2 (cumulative)
PARITY_REPORT, PARITY_DROP = cx.parity_report(ROOT, CTX_VERSION, TRAIN_C)
if PARITY_REPORT.empty:
    raise RuntimeError("parity report is empty - Block 4 context features for HOLDOUT / TEST are missing")
b5.mark(b5.P("parity.json"), {"drop": PARITY_DROP})
print("parity drop (KS > 0.10 or NaN-rate gap > 5pp, HOLDOUT vs TEST):", PARITY_DROP)
print(PARITY_REPORT.sort_values("ks", ascending=False).head(8)[["country", "feature", "ks", "nan_holdout", "nan_test"]].round(4).to_string(index=False))
if sorted(PARITY_DROP) != sorted(PARITY_EXPECTED_DROP):
    log(f"NOTE: the submitted run dropped {PARITY_EXPECTED_DROP}; this run drops {PARITY_DROP} - "
        "the model and the final file will differ slightly from the submitted ones")
FEATS0 = [f for f in FSET_L2 if f not in set(PARITY_DROP)]
log(f"{len(FEATS0)} features before the availability check ({len(B3F)} Block 3 + {len(FEATS0) - len(B3F)} Block 4)")

In [ ]:
# Stage 3 — hyper-parameters: tuned values if they exist, else the fixed defaults (optionally run the search)
SAFE = {"depth", "learning_rate", "l2_leaf_reg", "random_strength", "bagging_temperature", "border_count"}
HPO_BEST_P = b5.P("hpo", "best.json")


def _read_params(p):
    j = json.load(open(p))
    return {k: v for k, v in (j.get("params", j) if isinstance(j, dict) else {}).items() if k in SAFE}


def resolve_params():
    """First hit wins: CB_PARAMS > this run's HPO result > src/catboost_params.json >
    newest older block5/*/hpo/best.json > b5.CB_DEFAULT."""
    if CB_PARAMS:
        return {k: v for k, v in CB_PARAMS.items() if k in SAFE}, "CB_PARAMS in this notebook"
    older = sorted(glob.glob(f"{ROOT}/block5/*/hpo/best.json"), key=os.path.getmtime, reverse=True)
    shipped = SRC / "catboost_params.json"
    for p in [HPO_BEST_P, str(shipped) if shipped.exists() else None] + older:
        if p and os.path.exists(p):
            got = _read_params(p)
            if got:
                return got, p
    return {}, "b5.CB_DEFAULT (no tuned parameters found)"


PARAMS, PARAMS_SRC = resolve_params()
FIT = DEV = FEATS = None


def load_tree_data():
    """TREE-FIT / TREE-DEV with the feature list restricted to columns that exist; loaded once."""
    global FIT, DEV, FEATS
    if FIT is None:
        FIT = b5.load_role(b3, "train", "TREE-FIT", TRAIN_C)
        DEV = b5.load_role(b3, "train", "TREE-DEV", TRAIN_C)
        cols = set(FIT.columns) & set(DEV.columns)
        missing = [f for f in FEATS0 if f not in cols]
        if missing:
            log(f"features absent from the data (dropped): {missing}")
        FEATS = [f for f in FEATS0 if f in cols]


if not PARAMS and RUN_HPO:
    require("optuna")
    load_tree_data()
    SC_DEV = b5.scorer_for(ev, DEV)
    fit_h = FIT[ev.in_s1_subsample(FIT["s1_id"].to_numpy(), HPO_FIT_FRAC)]
    log(f"HPO: up to {HPO_MAX_TRIALS} trials / {HPO_MINUTES} min on {len(fit_h):,} TREE-FIT rows, scored on TREE-DEV "
        f"({len(FEATS)} features); trial 0 = the defaults")
    b5.hpo_worker(ev, b3, "A", fit_h, DEV, SC_DEV, FEATS, minutes=HPO_MINUTES, max_trials=HPO_MAX_TRIALS)
    _tab, _best = b5.hpo_merge(["A"])
    print(_tab.head(8)[["trial", "value", "best_iteration"]].to_string(index=False))
    del fit_h
    PARAMS, PARAMS_SRC = resolve_params()
if PARAMS:
    log("tuned CatBoost parameters:", PARAMS, "| from", PARAMS_SRC)
else:
    log("no tuned parameters found (set RUN_HPO = True to search) -> fixed defaults b5.CB_DEFAULT: "
        f"{ {k: v for k, v in b5.CB_DEFAULT.items() if k != 'iterations'} }")
PARAMS["iterations"] = ITER_CEILING

In [ ]:
# Stage 4 — train (TREE-FIT) with early stopping on TREE-DEV; skipped when a matching model is saved
from catboost import CatBoostClassifier
MODEL_P = b5.P("model", f"cb_seed{SEED}.cbm")
MODEL_META_P = b5.P("model", f"cb_seed{SEED}.json")
os.makedirs(os.path.dirname(MODEL_P), exist_ok=True)

SCORE_JOBS = [("train", "CALIBRATION", c) for c in TRAIN_C] + [("train", "HOLDOUT", c) for c in TRAIN_C] + \
             [("test", "TEST", c) for c in TEST_C]


def _done(sp, r, c):
    return os.path.exists(b5.score_path(sp, r, c).replace(".parquet", ".done.json"))


SAVED = False
if not FORCE_RETRAIN and os.path.exists(MODEL_P) and os.path.exists(MODEL_META_P):
    _m = json.load(open(MODEL_META_P))
    _sf = _m["feats"]
    # reuse the saved model only if it was trained with these parameters and this feature list
    if _m.get("params") == PARAMS and _sf == [f for f in FEATS0 if f in set(_sf)]:
        FEATS, SAVED = _sf, True
    else:
        log("a saved model exists but its parameters / features differ from this run's - training a new one")

if SAVED:
    MODEL = CatBoostClassifier().load_model(MODEL_P)
    log(f"loaded saved model {MODEL_P} (best iteration {MODEL.get_best_iteration()}, {len(FEATS)} features)")
else:
    t0 = time.time()
    load_tree_data()
    log(f"training on {len(FIT):,} TREE-FIT rows x {len(FEATS)} features; early stopping on {len(DEV):,} TREE-DEV rows")
    MODEL = b5.fit_cb(FIT, DEV, FEATS, CATS, PARAMS, seed=SEED)
    tmp = os.path.join(WORK, os.path.basename(MODEL_P))
    MODEL.save_model(tmp)
    shutil.copyfile(tmp, MODEL_P)
    jdump({"feats": FEATS, "params": PARAMS, "params_src": PARAMS_SRC, "task_type": CB_TASK_TYPE, "seed": SEED,
           "best_iteration": int(MODEL.get_best_iteration()), "n_fit_rows": int(len(FIT)),
           "parity_drop": PARITY_DROP, "code": [b5.B5_CODE_VERSION, cx.CTX_CODE_VERSION],
           "time": time.strftime("%F %T")}, MODEL_META_P)
    log(f"trained: best iteration {MODEL.get_best_iteration()} ({(time.time() - t0) / 60:.1f} min); saved {MODEL_P}")
    del FIT
    gc.collect()
    SC_DEV = b5.scorer_for(ev, DEV)
    s, t, _ = b5.s1_macro(ev, SC_DEV, DEV, b5.pred_cb(MODEL, DEV, FEATS, CATS))
    log(f"TREE-DEV S1-macro F0.5 (best global threshold {t:.4f}): {s:.6f}   [submitted run, L2, 20% subsample: 0.987105]")
    del DEV
    gc.collect()
jdump({"feats": FEATS}, b5.P("features_final.json"))

## Scoring
CALIBRATION and HOLDOUT (labelled INDIA + US) and TEST (INDIA, US, FRANCE) → one probability per candidate pair.
Scored in row chunks so a 20M-row country never needs a second copy in memory. One parquet + `.done.json` per
(split, role, country).

In [ ]:
# Stage 5 — score
KEEP = ["s1_id", "cand_id", "source", "country", "role", "label"]
CHUNK = 2_000_000


def predict_p(df):
    out = np.empty(len(df), np.float32)
    for i in range(0, len(df), CHUNK):
        out[i:i + CHUNK] = b5.pred_cb(MODEL, df.iloc[i:i + CHUNK], FEATS, CATS)
    return out


for sp, r, c in SCORE_JOBS:
    outp = b5.score_path(sp, r, c)
    donep = outp.replace(".parquet", ".done.json")
    if os.path.exists(donep) and not FORCE_RESCORE:
        log(f"score {sp}/{r}/{c}: committed, skip")
        continue
    t0 = time.time()
    df = b5.load_role(b3, sp, r, [c])
    out = df[[k for k in KEEP if k in df.columns]].copy()
    out["p"] = predict_p(df)
    del df
    gc.collect()
    b5.write_parquet(out, outp)
    b5.mark(donep, {"rows": int(len(out)), "n_s1": int(out["s1_id"].nunique()), "p_mean": float(out["p"].mean()),
                    "elapsed_s": round(time.time() - t0), "n_features": len(FEATS)})
    log(f"score {sp}/{r}/{c}: {len(out):,} rows, {out['s1_id'].nunique():,} S1, mean p {out['p'].mean():.4f} "
        f"({time.time() - t0:.0f}s)")
    del out
    gc.collect()

In [ ]:
# Stage 6 — summary
rows = []
for sp, r, c in SCORE_JOBS:
    j = json.load(open(b5.score_path(sp, r, c).replace(".parquet", ".done.json")))
    rows.append({"split": sp, "role": r, "country": c, "rows": j["rows"], "n_s1": j["n_s1"], "mean_p": round(j["p_mean"], 4)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nBlock 5 done. Next: 06_block6.ipynb")